# 6. Interactive Monte Carlo

- **Objective**: Dynamically manipulate parameters and instantly visualize their effects on both the path trajectories and final distribution.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
try:
    from ipywidgets import interact, FloatSlider, IntSlider, Dropdown
    widgets_available = True
except ImportError:
    widgets_available = False

def plot_interactive_mc(S0, mu, sigma, T, N_paths, distribution):
    N_steps = int(252 * T)
    dt = T / N_steps

    paths = np.zeros((N_steps, N_paths))
    paths[0] = S0

    for t in range(1, N_steps):
        if distribution == 'Normal':
            Z = np.random.standard_normal(N_paths)
        else: # 'Fat-Tailed' (Student T with df=3)
            Z = np.random.standard_t(3, size=N_paths)

        paths[t] = paths[t-1] * np.exp((mu - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * Z)

    ST = paths[-1, :]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

    # Path Plot
    ax1.plot(paths, lw=0.5, alpha=0.5)
    ax1.set_title(f'{N_paths} Simulated Paths over {T} Years')
    ax1.set_xlabel('Time Steps')
    ax1.set_ylabel('Price')

    # Histogram Plot
    ax2.hist(ST, bins=50, color='skyblue', edgecolor='black', density=True)
    ax2.axvline(ST.mean(), color='red', linestyle='dashed', linewidth=2, label=f'Mean: ${ST.mean():.2f}')
    ax2.set_title(f'Final Price Distribution ({distribution} Noise)')
    ax2.set_xlim(0, max(300, np.percentile(ST, 99)))
    ax2.legend()

    plt.show()

if widgets_available:
    interact(plot_interactive_mc,
             S0=FloatSlider(value=100, min=50, max=200, step=10),
             mu=FloatSlider(value=0.08, min=-0.1, max=0.2, step=0.01),
             sigma=FloatSlider(value=0.2, min=0.05, max=0.8, step=0.05),
             T=FloatSlider(value=1.0, min=0.5, max=5.0, step=0.5),
             N_paths=IntSlider(value=100, min=10, max=1000, step=50),
             distribution=Dropdown(options=['Normal', 'Fat-Tailed'], value='Normal'));
else:
    plot_interactive_mc(100, 0.08, 0.2, 1.0, 100, 'Normal')

interactive(children=(FloatSlider(value=100.0, description='S0', max=200.0, min=50.0, step=10.0), FloatSlider(…

## Questions for Understanding

- Toggle the distribution from 'Normal' to 'Fat-Tailed'. What visually happens to the simulated paths and the width of the final histogram?
- If you increase $T$ to 5 years, why does the final distribution become extremely skewed to the right (a long right tail)?